In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import pipeline
from sklearn.metrics import average_precision_score, ndcg_score




In [3]:
df = pd.read_csv("../data/vectors_hr_data.csv")
pd.set_option("display.max_colwidth", None)


In [4]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

BoW  is defined as how many times does the word appear?
TF-IDF is defined as how important/distinctive is this word compared with the rest of the dataset?

Each word has a GloVe vector → average all word vectors → get one vector representing the entire job title.

Among the evaluated embedding approaches, E5 produced the most relevant job-title rankings. This is consistent with E5's retrieval-oriented training, which is specifically designed to represent semantically related texts close together in the embedding space.

In [5]:
def ranked_by_model(similarity_model, col, relevance):
    ranked = df.sort_values(similarity_model, ascending=False)

    return ranked[[col, similarity_model, relevance]]

| **Precision@5** | Fraction of the first 5 candidates that are relevant.
| **Precision@10** | Fraction of the first 10 candidates that are relevant. 
| **MRR** | How early the first relevant candidate appears. 
| **AP** | Looks across the entire ranking and rewards putting relevant candidates before irrelevant ones. 
| **NDCG@10** | actual ranking score / best possible ranking score

In [6]:
def metrics(ranked, relevance, similarity_model):
    labels = ranked[relevance].to_numpy()
    scores = ranked[similarity_model].to_numpy()

    precision_5 = labels[:5].sum() / 5
    precision_10 = labels[:10].sum() / 10

    relevant_positions = np.where(labels == 1)[0] + 1
    mrr = 1 / relevant_positions[0] if len(relevant_positions) else 0

    if labels.sum() > 0:
        ap = average_precision_score(labels, scores)
        ndcg_10 = ndcg_score(
            labels.reshape(1, -1),
            scores.reshape(1, -1),
            k=10
        )
    else:
        ap = 0
        ndcg_10 = 0

    return {
        "Precision@5": precision_5,
        "Precision@10": precision_10,
        "MRR": mrr,
        "AP": ap,
        "NDCG@10": ndcg_10
    }


In [7]:

bow_ranked = ranked_by_model("similarity_1_bow","clean_job_title","relevance")
bow_metrics = metrics(bow_ranked, "relevance","similarity_1_bow")

tfidf_ranked = ranked_by_model("similarity_1_tfidf","clean_job_title","relevance")
tfidf_metrics = metrics(tfidf_ranked, "relevance","similarity_1_tfidf")

glove_ranked = ranked_by_model("similarity_1_glove","clean_job_title","relevance")
glove_metrics = metrics(glove_ranked, "relevance","similarity_1_glove")

bert_ranked = ranked_by_model("similarity_1_bert","clean_job_title","relevance")
bert_metrics = metrics(bert_ranked, "relevance","similarity_1_bert")

sbert_ranked = ranked_by_model("similarity_1_sbert","clean_job_title","relevance")
sbert_metrics = metrics(sbert_ranked, "relevance","similarity_1_sbert")

e5_ranked = ranked_by_model("similarity_1_5e","clean_job_title","relevance")
e5_metrics = metrics(e5_ranked, "relevance","similarity_1_5e")

bge_ranked = ranked_by_model("similarity_1_bge","clean_job_title","relevance")
bge_metrics = metrics(bge_ranked, "relevance","similarity_1_bge")

In [8]:
bow_ranked = ranked_by_model("similarity_2_bow","clean_job_title","relevance")
bow_metrics = metrics(bow_ranked, "relevance","similarity_2_bow")

tfidf_ranked = ranked_by_model("similarity_2_tfidf","clean_job_title","relevance")
tfidf_metrics = metrics(tfidf_ranked, "relevance","similarity_2_tfidf")

glove_ranked = ranked_by_model("similarity_2_glove","clean_job_title","relevance")
glove_metrics = metrics(glove_ranked, "relevance","similarity_2_glove")

bert_ranked = ranked_by_model("similarity_2_bert","clean_job_title","relevance")
bert_metrics = metrics(bert_ranked, "relevance","similarity_2_bert")

sbert_ranked = ranked_by_model("similarity_2_sbert","clean_job_title","relevance")
sbert_metrics = metrics(sbert_ranked, "relevance","similarity_2_sbert")

e5_ranked = ranked_by_model("similarity_2_5e","clean_job_title","relevance")
e5_metrics = metrics(e5_ranked, "relevance","similarity_2_5e")

bge_ranked = ranked_by_model("similarity_2_bge","clean_job_title","relevance")
bge_metrics = metrics(bge_ranked, "relevance","similarity_2_bge")

In [9]:
results = pd.DataFrame({
    "Bow": bow_metrics,
    "TF-IDF": tfidf_metrics,
    "GloVe": glove_metrics,
    "BERT": bert_metrics,
    "SBERT": sbert_metrics,
    "E5": e5_metrics,
    "BGE": bge_metrics
}).T.rename_axis("Model").reset_index()

results

,Model,Precision@5,Precision@10,MRR,AP,NDCG@10
0,Bow,1.0,1.0,1.0,0.977855,1.000000
1,TF-IDF,1.0,1.0,1.0,0.977855,1.000000
2,GloVe,1.0,1.0,1.0,0.955214,1.000000
3,BERT,0.8,0.7,1.0,0.831193,0.753449
4,SBERT,1.0,1.0,1.0,0.995338,1.000000
5,E5,1.0,1.0,1.0,0.990001,1.000000
6,BGE,1.0,1.0,1.0,0.987924,1.000000


In [ ]:
X_e5 = np.vstack(df["e5_for_modeling"])

star_candidate = X_e5[6] # student humber college aspiring human resource

starred_similarity = cosine_similarity(
    X_e5,
    star_candidate.reshape(1, -1)
).flatten()

alpha = 0.6

df['rerank_score'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["all_data", "rerank_score"]].sort_values(
    "rerank_score",
    ascending=False
).head(10)

NameError: name 'X_e5' is not defined

In [ ]:
star_candidates = X_e5[2:6] # student number college aspiring human resource
star_anchor = star_candidates.mean(axis=0)
starred_similarity = cosine_similarity(
    X_e5,
    star_anchor.reshape(1, -1)
).flatten()

alpha = 0.5

df['rerank_scores'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["job_title", "rerank_scores"]].sort_values(
    "rerank_scores",
    ascending=False
).head(10)

,job_title,rerank_scores
27,Seeking Human Resources Opportunities,0.851140
2,Aspiring Human Resources Professional,0.848382
9,Seeking Human Resources HRIS and Generalist Positions,0.836446
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.829493
5,Aspiring Human Resources Specialist,0.828090
98,Seeking Human Resources Position,0.824765
75,Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment,0.811959
26,Aspiring Human Resources Management student seeking an internship,0.810844
93,Seeking Human Resources Opportunities. Open to travel and relocation.,0.808721
73,Human Resources Professional,0.804489


: 

In [11]:
def zero_shot_classifer_check():
    zero_shot_classifier = pipeline(
        task="zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

    candidate_labels = [
        "This candidate HR related",
        "This candidate not HR related"
    ]

    texts = df["clean_job_title"].tolist()

    results = zero_shot_classifier(
        texts,
        candidate_labels=candidate_labels
    )

    return results

In [ ]:
results = zero_shot_classifer_check()
df_results = pd.DataFrame(results)
df_results

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

I examined the distribution of candidate locations using a count plot. Some locations were more represented than others, but this does not demonstrate ranking bias. Because location and connection count were included in the combined text used for embeddings, they may influence similarity scores. Further evaluation would compare rankings generated from job titles alone with rankings generated from the combined text.